In [1]:
import numpy as np
import os
import time
from maxclique import read_dimacs_clq, build_clique_hamiltonian
from maxclique import clique_number_from_energy, extract_clique
from maxclique import greedy_clique_from_weights

### Define multistart projected gradient descent

In [2]:
def project_simplex(v, sum_constraint):
    """Exact Euclidean projection onto {x >= 0, sum(x) == R}.
    """
    R = float(sum_constraint)
    v = np.asarray(v, dtype=float)
    u = np.sort(v)[::-1]
    cumulative = np.cumsum(u) - R
    ind = np.arange(1, v.size + 1)
    rho = ind[u - cumulative / ind > 0][-1]
    return np.maximum(v - cumulative[rho - 1] / rho, 0.0)


def pgd(Q, c, sum_constraint, lr=0.01, max_iter=10**4, tol=1e-9, x0=None):
    """Projected gradient descent on {x >= 0, sum(x) == R}.
    """
    Q = np.asarray(Q, dtype=float)
    c = np.asarray(c, dtype=float).ravel()
    n = Q.shape[0]
    R = float(sum_constraint)

    x = np.full(n, R / n) if x0 is None else np.asarray(x0, dtype=float)
    x = project_simplex(x, R)
    QT = Q + Q.T                                   # gradient of x'Qx is (Q+Q')x

    it = 0
    for it in range(max_iter):
        grad = QT @ x + c
        x_next = project_simplex(x - lr * grad, R)
        delta = np.linalg.norm(x_next - x)
        x = x_next                                 # keep the newer iterate
        if delta < tol:
            break

    return x, float(x @ Q @ x + c @ x), it + 1


def pgd_multistart(Q, c, sum_constraint, restarts=32, seed=0, **kwargs):
    """PGD from the simplex centre plus random starts.
    """
    rng = np.random.default_rng(seed)
    n = Q.shape[0]
    solutions, energies, times, total_iters = [], [], [], 0
    for r in range(restarts):
        start = time.time()
        x0 = None
        if r > 0:
            x0 = rng.random(n)
            x0 *= sum_constraint / x0.sum()
        x, energy, iters = pgd(Q, c, sum_constraint, x0=x0, **kwargs)
        pgd_t = time.time()-start
        solutions.append(x)
        energies.append(energy)
        times.append(pgd_t)
        total_iters += iters
    return solutions, energies, times, total_iters


### load instance file and solve

In [5]:
instance_dir = "Instances/"
instance_name = "keller4.clq"
instance_path = os.path.join(instance_dir, instance_name)
try:
    A, n, m = read_dimacs_clq(instance_path)
    print("file loaded sucessfully")
except FileNotFoundError:
    print(f"{instance_path} does not exist")

file loaded sucessfully


In [6]:
name = os.path.splitext(instance_name)[0]
print(f"{name}: n = {n} vertices, m = {m} edges (declared)")
print(f"A shape: {A.shape}")
print(f"edges in A: {int(A.sum()) // 2}")
print(f"symmetric: {np.array_equal(A, A.T)}")
print(f"self-loops: {int(np.trace(A))}")
print(f"density: {A.sum() / (n * (n - 1)):.5f}")


keller4: n = 171 vertices, m = 9435 edges (declared)
A shape: (171, 171)
edges in A: 9435
symmetric: True
self-loops: 0
density: 0.64912


In [7]:
c = np.zeros(n)
H = build_clique_hamiltonian(A)

sum_constraint = 1
restarts = 100
learning_rate = 0.01
max_iter = 2 * 10**4

In [10]:
pgd_start = time.time()
solutions, energies, times, total_iters = pgd_multistart(
    Q=H,
    c=c,
    sum_constraint=sum_constraint,
    restarts=restarts,
    lr=learning_rate,
    max_iter=max_iter,
)
pgd_time = time.time() - pgd_start

In [11]:
best_energy = min(energies)
best_solution = solutions[int(np.argmin(energies))]
omega = clique_number_from_energy(best_energy, sum_constraint)
support, isclique = extract_clique(best_solution, A)

# Round every restart, not just the lowest-energy one.
cliques = [greedy_clique_from_weights(sol, A) for sol in solutions]
greedy = max(cliques, key=len)


In [12]:

print(f"restarts:{restarts}, total PGD iterations:{total_iters}")
print(f"best energy over restarts:{best_energy:.6f}")
print(f"support size:{len(support)}, support is a clique:{isclique}")
print(f"clique vertices (1-indexed):{(greedy + 1).tolist()}")
print(f"{'energy':>16}{'omega_est':>12}{'clique':>9}{'time (s)':>12}")
print(f"{best_energy:>16.6f}{omega:>12.3f}{len(greedy):>9}{pgd_time:>12.3f}")

# Never report a set that is not actually a clique.
assert extract_clique(np.isin(np.arange(n), greedy).astype(float), A)[1], \
    "greedy repair returned a non-clique"


restarts:100, total PGD iterations:77765
best energy over restarts:-0.888889
support size:15, support is a clique:False
clique vertices (1-indexed):[3, 24, 28, 37, 59, 77, 87, 105, 111]
          energy   omega_est   clique    time (s)
       -0.888889       9.000        9       0.735
